## [B]Nutritional content of different food

We first paste in our fdc api key and install our dependencies via requirements.txt:

In [63]:
apikey = "6k4anDq3cih8Tre4oqT28vAxd5dKKOAfKm53hdWx"

In [64]:
%pip install -r ../requirements.txt --upgrade -q
!pip install fooddatacentral
!pip install pint

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'
Note: you may need to restart the kernel to use updated packages.


Import the necessary packages for pulling and processing data:

In [65]:
import pandas as pd
import fooddatacentral as fdc


In [66]:
from  scipy.optimize import linprog as lp
import numpy as np
import warnings

import pandas as pd
import numpy as np
import warnings
from scipy.optimize import linprog as lp

import pandas as pd
import numpy as np
import warnings
from scipy.optimize import linprog as lp

def solve_subsistence_problem(FoodNutrients, Prices, dietmin, max_weight=None, tol=1e-6, drop_vitamins=None):
    """
    Solves Stigler's Subsistence Cost Problem using minimum requirements.
    """
    
    vitamin_map = {
        'A': 'Vitamin A, RAE',
        'E': 'Vitamin E (alpha-tocopherol)',
        'C': 'Vitamin C, total ascorbic acid'
    }

    if drop_vitamins is not None:
        if isinstance(drop_vitamins, str):
            drop_vitamins = [drop_vitamins]
            
        labels_to_drop = [vitamin_map[v.upper()] for v in drop_vitamins if v.upper() in vitamin_map]
        
        dietmin = dietmin.drop(labels=labels_to_drop, errors='ignore')

    
    try: 
        p = Prices.apply(lambda x: x.magnitude)
    except AttributeError: 
        p = Prices

    p = p.dropna()
    use = p.index.intersection(FoodNutrients.columns)
    p = p[use]

    A = FoodNutrients[p.index].fillna(0)
    A = A.loc[A.index.intersection(dietmin.index)]
    A = A.reindex(dietmin.index, axis=0)
    b = dietmin

    A = A.reindex(p.index, axis=1)
    A = A.reindex(b.index, axis=0)

    result = lp(p.values, -A.values, -b.values, method='highs')

    result.A = A
    result.b = b
    
    if result.success:
        result.diet = pd.Series(result.x, index=p.index)
    else:
        warnings.warn(result.message)
        result.diet = pd.Series(np.nan, index=p.index)

    return result
    
   

   

In [67]:
dfNG = pd.DataFrame({
    "Food": [
        "Malt drinks",
        "Soft drinks (Coca Cola, spirit, etc)",
        "Plantains",
        "Yam--roots",
        "Milk powder",
        "Chicken",
        "Condiments",
        "Tea",
        "Sachet water",
        "Palm oil",
        "Chocolate drinks",
        "Milk tinned (unsweetened)",
        "Tomato puree (canned)",
        "Fish--frozen",
        "Eggs",
        "Orange/tangerine",
        "Bread",
        "Groundnut oil",
        "Beef",
        "Goat",
        "Fish--dried",
        "Maize flour",
        "Rice--local",
        "Sweet potatoes",
        "Fish--smoked",
        "Millet",
        "Gari",
        "Sugar",
        "Maize",
        "Onions",
        "Groundnuts",
        "Fish--fresh",
        "Cassava (root)",
        "White beans",
        "Pepper",
        "Bananas",
        "Guinea corn/sorghum",
        "Cassava flour",
        "Garden eggs/egg plant",
        "Tomatoes",
        "Other vegetables (fresh or canned)",
        "Rice--imported",
        "Brown beans",
        "Okra",
        "Leaves (Cocoyam, Spinach, etc.)"
    ]
})

In [68]:
def get_fdc_id(food_name):
    try:
        results = fdc.search(apikey, food_name)

        if results is not None and len(results) > 0:
            return results.iloc[0]["fdcId"]   # not results[0]["fdcId"]
        else:
            return None

    except Exception as e:
        print(f"Error for {food_name}: {e}")
        return None

In [69]:
dfNG["FDC"] = dfNG["Food"].apply(get_fdc_id)
dfNG["FDC"] = dfNG["FDC"].astype("Int64")
dfNG.head()

,Food,FDC
0,Malt drinks,174866
1,"Soft drinks (Coca Cola, spirit, etc)",2710541
2,Plantains,2709563
3,Yam--roots,170071
4,Milk powder,2673399


In [70]:
nutrient_df = fdc.nutrients(apikey, "2673399")
nutrient_df

,Quantity,Units
"Sodium, Na",417.00,mg
"Fatty acids, total saturated",0.00,g
Energy,375.00,kcal
"Potassium, K",1708.00,mg
Total lipid (fat),0.00,g
"Fatty acids, total trans",0.00,g
"Iron, Fe",0.00,mg
"Calcium, Ca",1292.00,mg
Cholesterol,0.00,mg
"Sugars, added",0.00,g


In [71]:
dfNG["FDC"] = dfNG["FDC"].astype("Int64")
dfNG.head()

,Food,FDC
0,Malt drinks,174866
1,"Soft drinks (Coca Cola, spirit, etc)",2710541
2,Plantains,2709563
3,Yam--roots,170071
4,Milk powder,2673399


In [72]:
food_nutrient_dict = {}
failed_foods = []

for _, row in dfNG.iterrows():
    food = row["Food"]
    fdc_code = row["FDC"]

    try:
        if pd.notna(fdc_code):
            # IMPORTANT FIX: string + no decimals
            nutrient_df = fdc.nutrients(apikey, str(int(fdc_code)))

            if nutrient_df is not None and "Quantity" in nutrient_df.columns:
                food_nutrient_dict[food] = nutrient_df["Quantity"]
            else:
                failed_foods.append(food)
        else:
            failed_foods.append(food)

    except Exception as e:
        print(f"{food} failed: {e}")
        failed_foods.append(food)

In [73]:
FoodNutrientsNG = pd.DataFrame(food_nutrient_dict)
FoodNutrientsNG_T = FoodNutrientsNG.T

FoodNutrientsNG_T.head()

,Alanine,"Alcohol, ethyl",Amino acids,Arginine,Ash,Aspartic acid,Betaine,Caffeine,"Calcium, Ca","Carbohydrate, by difference",...,Vitamin D2 (ergocalciferol),Vitamin D3 (cholecalciferol),Vitamin E (alpha-tocopherol),"Vitamin E, added",Vitamin K (Dihydrophylloquinone),Vitamin K (Menaquinone-4),Vitamin K (phylloquinone),Vitamins and Other Components,Water,"Zinc, Zn"
Malt drinks,NaN,0.0,0.0,NaN,1.90,NaN,NaN,37.0,60.0,86.94,...,NaN,NaN,0.17,0.0,NaN,NaN,2.5,0.0,1.30,0.80
"Soft drinks (Coca Cola, spirit, etc)",NaN,0.0,NaN,NaN,NaN,NaN,NaN,9.0,1.0,10.40,...,NaN,NaN,0.00,0.0,NaN,NaN,0.0,NaN,89.40,0.09
Plantains,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,9.0,63.80,...,NaN,NaN,5.04,0.0,NaN,NaN,28.6,NaN,2.09,0.37
Yam--roots,0.063,0.0,0.0,0.127,0.82,0.155,NaN,0.0,17.0,27.88,...,NaN,NaN,0.35,0.0,NaN,NaN,2.3,0.0,69.60,0.24
Milk powder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1292.0,54.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [74]:
FoodNutrientsNG_T.to_csv("FoodNutrientsNG.csv")

In [75]:
wanted_nutrients = [
    "Energy",
    "Protein",
    "Total lipid (fat)",
    "Carbohydrate, by difference",
    "Fiber, total dietary",
    "Calcium, Ca",
    "Iron, Fe",
    "Potassium, K",
    "Sodium, Na"
]

clean_sheet = FoodNutrientsNG_T[wanted_nutrients]
clean_sheet

,Energy,Protein,Total lipid (fat),"Carbohydrate, by difference","Fiber, total dietary","Calcium, Ca","Iron, Fe","Potassium, K","Sodium, Na"
Malt drinks,1720.0,5.100000,4.760,86.940000,4.800,60.00,2.280,618.0,190.00
"Soft drinks (Coca Cola, spirit, etc)",42.0,0.000000,0.250,10.400000,0.000,1.00,0.020,5.0,3.00
Plantains,531.0,2.280000,29.600,63.800000,3.500,9.00,0.970,786.0,202.00
Yam--roots,494.0,1.530000,0.170,27.880000,4.100,17.00,0.540,816.0,9.00
Milk powder,375.0,33.330000,0.000,54.170000,4.200,1292.00,0.000,1708.0,417.00
Chicken,188.0,24.710000,9.410,1.180000,0.000,0.00,0.420,NaN,353.00
Condiments,1136.0,16.960000,16.220,14.430000,1.000,106.00,2.180,242.0,480.00
Tea,34.0,0.000000,0.000,8.460000,NaN,NaN,NaN,NaN,2.00
Sachet water,0.0,0.000000,0.000,0.000000,0.000,3.00,0.000,0.0,4.00
Palm oil,3699.0,0.000000,100.000,0.000000,0.000,0.00,0.010,0.0,0.00
